# 02 - Modeles recents (Transformers)

**Objectif** : Fine-tuner les modeles recents et evaluer leurs performances sur le test set.

## Modeles
1. DistilBERT (distilbert-base-uncased) -- BERT distille, plus leger
2. ModernBERT (answerdotai/ModernBERT-base) -- dec 2024
3. NeoBERT (neobert-base) -- feb 2025

## Metriques
- ROC-AUC (macro)
- F1-score (macro)
- Precision / Recall (macro)
- Hamming Loss
- Temps d'entrainement / inference

In [1]:
import pandas as pd
import numpy as np
import time
import json
import os
import warnings

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score, hamming_loss
)

warnings.filterwarnings('ignore')

LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
DATA_DIR = '../data/processed/'
MODELS_DIR = '../artifacts/models/'
METRICS_DIR = '../artifacts/metrics/'

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device : {device}")

Device : cuda


## 1. Chargement des donnees

In [2]:
train_df = pd.read_csv(DATA_DIR + 'train.csv')
val_df = pd.read_csv(DATA_DIR + 'val.csv')
test_df = pd.read_csv(DATA_DIR + 'test.csv')

print(f"Train : {len(train_df)}")
print(f"Val   : {len(val_df)}")
print(f"Test  : {len(test_df)}")

X_train = train_df['comment_text'].values
X_val = val_df['comment_text'].values
X_test = test_df['comment_text'].values

y_train = train_df[LABEL_COLS].values
y_val = val_df[LABEL_COLS].values
y_test = test_df[LABEL_COLS].values

Train : 111478
Val   : 23888
Test  : 23889


## 2. Classes et fonctions utilitaires

In [3]:
class ToxicDataset(Dataset):
    """Dataset PyTorch pour la classification multi-label."""

    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item


def compute_metrics_hf(eval_pred):
    """Fonction de metriques pour le Trainer HuggingFace."""
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs > 0.5).astype(int)
    return {
        'roc_auc_macro': roc_auc_score(labels, probs, average='macro'),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
    }


def evaluate_model(y_true, y_pred, y_prob, model_name, train_time, inference_time):
    """Evalue un modele et retourne un dictionnaire de metriques."""
    metrics = {
        'model': model_name,
        'roc_auc_macro': roc_auc_score(y_true, y_prob, average='macro'),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'train_time_sec': round(train_time, 2),
        'inference_time_sec': round(inference_time, 4),
    }
    for i, label in enumerate(LABEL_COLS):
        metrics[f'roc_auc_{label}'] = roc_auc_score(y_true[:, i], y_prob[:, i])

    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(f"  ROC-AUC (macro) : {metrics['roc_auc_macro']:.4f}")
    print(f"  F1 (macro)      : {metrics['f1_macro']:.4f}")
    print(f"  Precision       : {metrics['precision_macro']:.4f}")
    print(f"  Recall          : {metrics['recall_macro']:.4f}")
    print(f"  Hamming Loss    : {metrics['hamming_loss']:.4f}")
    print(f"  Train time      : {metrics['train_time_sec']:.2f}s")
    print(f"  Inference time  : {metrics['inference_time_sec']:.4f}s")
    return metrics


def finetune_and_evaluate(model_id, model_display_name, save_dir, max_length=256):
    """Fine-tune un modele Transformer et l'evalue sur le test set."""
    print(f"\n{'#'*60}")
    print(f"  {model_display_name} ({model_id})")
    print(f"{'#'*60}")

    # Tokenizer et datasets
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    train_dataset = ToxicDataset(X_train, y_train, tokenizer, max_length=max_length)
    val_dataset = ToxicDataset(X_val, y_val, tokenizer, max_length=max_length)
    test_dataset = ToxicDataset(X_test, y_test, tokenizer, max_length=max_length)

    # Modele
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=len(LABEL_COLS),
        problem_type='multi_label_classification'
    )

    training_args = TrainingArguments(
        output_dir=save_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='roc_auc_macro',
        greater_is_better=True,
        logging_steps=100,
        seed=42,
        fp16=torch.cuda.is_available(),
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics_hf,
    )

    # Entrainement
    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0

    # Evaluation sur test
    t0 = time.time()
    predictions = trainer.predict(test_dataset)
    inference_time = time.time() - t0

    logits = predictions.predictions
    y_prob = torch.sigmoid(torch.tensor(logits)).numpy()
    y_pred = (y_prob > 0.5).astype(int)

    metrics = evaluate_model(y_test, y_pred, y_prob, model_display_name, train_time, inference_time)

    # Sauvegarde
    best_dir = os.path.join(save_dir, 'best')
    trainer.save_model(best_dir)
    tokenizer.save_pretrained(best_dir)
    print(f"  Modele sauvegarde dans {best_dir}")

    return metrics


all_results = []

## 3. DistilBERT

In [4]:
results_distilbert = finetune_and_evaluate(
    model_id='distilbert-base-uncased',
    model_display_name='DistilBERT',
    save_dir=MODELS_DIR + 'distilbert/',
    max_length=256
)
all_results.append(results_distilbert)


############################################################
  DistilBERT (distilbert-base-uncased)
############################################################


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Roc Auc Macro,F1 Macro
1,0.040850,0.037094,0.990543,0.636897
2,0.033180,0.037269,0.991299,0.660685
3,0.023132,0.039718,0.991432,0.672467


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



  DistilBERT
  ROC-AUC (macro) : 0.9895
  F1 (macro)      : 0.6579
  Precision       : 0.7005
  Recall          : 0.6288
  Hamming Loss    : 0.0153
  Train time      : 1566.89s
  Inference time  : 33.4342s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Modele sauvegarde dans ../artifacts/models/distilbert/best


## 4. ModernBERT

In [5]:
results_modernbert = finetune_and_evaluate(
    model_id='answerdotai/ModernBERT-base',
    model_display_name='ModernBERT',
    save_dir=MODELS_DIR + 'modernbert/',
    max_length=256
)
all_results.append(results_modernbert)


############################################################
  ModernBERT (answerdotai/ModernBERT-base)
############################################################


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Roc Auc Macro,F1 Macro
1,0.039578,0.036832,0.991404,0.656512
2,0.029619,0.037888,0.991433,0.659277
3,0.014433,0.049185,0.987947,0.647007


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ModernBERT
  ROC-AUC (macro) : 0.9911
  F1 (macro)      : 0.6632
  Precision       : 0.7081
  Recall          : 0.6350
  Hamming Loss    : 0.0153
  Train time      : 4835.54s
  Inference time  : 120.1798s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Modele sauvegarde dans ../artifacts/models/modernbert/best


## 5. NeoBERT

In [6]:
results_neobert = finetune_and_evaluate(
    model_id='chandar-lab/NeoBERT',
    model_display_name='NeoBERT',
    save_dir=MODELS_DIR + 'neobert/',
    max_length=256
)
all_results.append(results_neobert)


############################################################
  NeoBERT (chandar-lab/NeoBERT)
############################################################


config.json:   0%|          | 0.00/928 [00:00<?, ?B/s]

model.py: 0.00B [00:00, ?B/s]

Encountered exception while importing xformers: No module named 'xformers'
You are using a model of type `neobert` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Encountered exception while importing xformers: No module named 'xformers'


ImportError: This modeling file requires the following packages that were not found in your environment: xformers. Run `pip install xformers`

## 6. Sauvegarde et resume

In [ ]:
# Sauvegarder les metriques
with open(METRICS_DIR + 'transformers_metrics.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("Metriques sauvegardees dans", METRICS_DIR + 'transformers_metrics.json')

In [ ]:
# Resume comparatif des modeles recents
results_df = pd.DataFrame(all_results)
cols = ['model', 'roc_auc_macro', 'f1_macro', 'precision_macro', 'recall_macro',
        'hamming_loss', 'train_time_sec', 'inference_time_sec']
print("\n" + "=" * 80)
print("RESUME DES MODELES RECENTS")
print("=" * 80)
print(results_df[cols].to_string(index=False))
print("\nProchaine etape : notebook 03_comparaison.ipynb")